# Aula 25/03

Implementação Algoritmo genético da função g11 => f(x) = x1² + (x2 − 1)² com a restrição => (x2 - x1²) = 0

intervalo: −1 ≤ x1 ≤ 1 e −1 ≤ x2 ≤ 1. \
A solução ótima é: x∗ = (−0.707036070037170616,0.500000004333606807) \
onde f(x∗) = 0.7499.


In [283]:
# f(x) = x1² + (x2 − 1)²
def g11(x1, x2):
    return x1**2 + (x2 - 1)**2

# restrição: (x2 - x1²) = 0
def h1(x1, x2):
    return abs(x2 - x1**2)

def fitness(x1, x2):
  y = g11(x1, x2)
  y_h1 = h1(x1, x2)

  if y_h1 > 10e-6:
      y += y_h1

  return(x1, x2, y)

In [316]:
import random
def gerar_populacao_inicial():
  populacao = []
  for x in range(10):
    x1 = random.uniform(-1, 1)
    x2 = random.uniform(-1, 1)
    tupla_individuo = fitness(x1, x2)
    populacao.append(tupla_individuo)

  return populacao

print(gerar_populacao_inicial())

[(0.3321514636921399, 0.47433435652666023, 0.7506587252549006), (0.381077570014416, -0.877145425005954, 4.691260600362922), (0.9216124169050133, -0.28461563293750425, 3.6335918513119285), (0.02797409263713435, -0.4398964720052545, 2.514763421816175), (0.10175715046967282, -0.8399791913147872, 4.24621165112962), (-0.7578297005780206, -0.15472907878532416, 2.636740034334072), (0.006360061565775066, 0.3637102681352926, 0.7685748910117538), (0.7754338795655713, 0.5151920149702396, 0.9224421705346026), (-0.5337179933909328, 0.20034699492425845, 1.0088077265408932), (-0.9738360845711171, -0.5420676924798107, 4.816753899895426)]


In [317]:
# Real para Binário
def real_para_binario(x, min_val, max_val, bits=12):
    max_int = (2 ** bits) - 1 #maior número representável (ex: 12 bits => 4095)
    inteiro = int((x - min_val) / (max_val - min_val) * max_int)
    return format(inteiro, f'0{bits}b') #formata o numero em uma string de binario, com a quantidade de bits

# Binário para Real
def binario_para_real(bin_str, min_val, max_val):
    inteiro = int(bin_str, 2)
    max_int = (2 ** len(bin_str)) - 1
    x = min_val + (inteiro / max_int) * (max_val - min_val)
    return x

print("Real 0.285 em binário: ", real_para_binario(0.258, -1, 1))
print("Binario para real: ", binario_para_real(real_para_binario(0.258, -1, 1), -1, 1))


Real 0.285 em binário:  101000001111
Binario para real:  0.2576312576312576


In [318]:
def converter_populacao_para_binario(populacao, min_val=-1, max_val=1, bits=12):
    populacao_bin = []

    for x1, x2, y in populacao:
        bin_x1 = real_para_binario(x1, min_val, max_val, bits)
        bin_x2 = real_para_binario(x2, min_val, max_val, bits)

        populacao_bin.append((bin_x1, bin_x2, y))

    return populacao_bin
print(converter_populacao_para_binario(gerar_populacao_inicial()))

[('111110101011', '111011010100', 1.0069118406809001), ('101010111000', '100011100010', 0.9106126458221468), ('100000110100', '001011101100', 3.3069554812640782), ('100011101111', '011001001101', 1.709114124462051), ('011111101111', '000110000000', 4.096992368352487), ('011010101111', '100010001010', 0.936915848071198), ('001101010100', '000110101101', 4.675977136726826), ('011110100001', '110010001000', 0.7544980583690681), ('010101101101', '101000111110', 0.7979954938033105), ('010100111001', '010010000011', 2.737039111547581)]


In [319]:
def torneio(pop_bin): #sorteia o pai - retorna a tupla (x1, x2, y)
  ind1, ind2 = random.sample(pop_bin, 2)

  if ind1[2] < ind2[2]:
      return ind1
  else:
      return ind2

In [320]:
def cruzamento(pai1, pai2):
    x1_pai1, x2_pai1, _ = pai1
    x1_pai2, x2_pai2, _ = pai2

    ponto = len(x1_pai1) // 2 # ponto (metade = 6)

    # filho1
    # 6 bits iniciais do pai1 + 6 bits finais do pai2
    filho1_x1 = x1_pai1[:ponto] + x1_pai2[ponto:]
    filho1_x2 = x2_pai1[:ponto] + x2_pai2[ponto:]

    # filho2
    # 6 bits iniciais do pai2 + 6 bits finais do pai1
    filho2_x1 = x1_pai2[:ponto] + x1_pai1[ponto:]
    filho2_x2 = x2_pai2[:ponto] + x2_pai1[ponto:]

    # filhos sem fitness(y)
    filho1 = (filho1_x1, filho1_x2, None)
    filho2 = (filho2_x1, filho2_x2, None)

    return filho1, filho2

In [321]:
def mutacao(pop, taxa=0.3):
    nova_pop = []

    for x1_bin, x2_bin, y in pop:

        # decide se haverá mutacao ou nao
        if random.random() < taxa:

            # escolhe qual variável mutar
            if random.random() < 0.5:
                # muta x1
                pos = random.randint(0, len(x1_bin) - 1)
                bit = x1_bin[pos]
                novo_bit = '1' if bit == '0' else '0'
                x1_bin = x1_bin[:pos] + novo_bit + x1_bin[pos+1:]

            else:
                # muta x2
                pos = random.randint(0, len(x1_bin) - 1)
                bit = x2_bin[pos]
                novo_bit = '1' if bit == '0' else '0'
                x2_bin = x2_bin[:pos] + novo_bit + x2_bin[pos+1:]

        nova_pop.append((x1_bin, x2_bin, y))

    return nova_pop


In [322]:
def gerar_filhos(pop_bin):
    nova_pop = []

    for _ in range(5): # a cada iteracao gera 2 filhos
        pai1 = torneio(pop_bin)
        pai2 = torneio(pop_bin)
        filho1, filho2 = cruzamento(pai1, pai2)

        nova_pop.append(filho1)
        nova_pop.append(filho2)

    return nova_pop

In [333]:
def calcular_fitness(pop_bin, min_val=-1, max_val=1):
    nova_pop = []

    for x1_bin, x2_bin, _ in pop_bin:
        # binário → real
        x1 = binario_para_real(x1_bin, min_val, max_val)
        x2 = binario_para_real(x2_bin, min_val, max_val)

        # calcula fitness
        x1_real, x2_real, y = fitness(x1, x2)
        nova_pop.append((x1_bin, x2_bin, y))

    return nova_pop

In [ ]:
def main():

  pop_inicial = gerar_populacao_inicial()
  pop_bin = converter_populacao_para_binario(pop_inicial)
  print(pop_inicial)
  for geracao in range(30):
      filhos = gerar_filhos(pop_bin)
      filhos_mutados = mutacao(filhos)
      pop_bin = calcular_fitness(filhos_mutados)
      melhor = min(pop_bin, key=lambda ind: ind[2])
      print(f"Geração {geracao}: melhor y = {melhor[2]}")

main()

[(-0.9137586639851698, 0.8352085199251169, 0.8623647518303875), (-0.8301621973543802, -0.5422795554738371, 4.29924433053892), (-0.1854361220286671, 0.49663129378202253, 0.7500113481815831), (0.8452427431630487, -0.16035683286079028, 2.9356554021671055), (-0.5669055185678553, -0.8775619548583846, 5.045564583155401), (-0.3538273054866474, -0.4747726471078504, 2.9001145319812296), (-0.7256967747070766, 0.7769298410321619, 0.8266901368540984), (0.9372772336095654, 0.3109705627889292, 1.9207682278400753), (0.8690687094361371, -0.10806007397716955, 2.846418044961446), (-0.19493940646092733, 0.7339074530232392, 0.8047126965798189)]
Geração 0: melhor y = 0.7500357951640003
Geração 1: melhor y = 0.7500125379978859
Geração 2: melhor y = 0.7500065746219593
Geração 3: melhor y = 0.7500065746219593
Geração 4: melhor y = 0.7500025195263291
Geração 5: melhor y = 0.7500025195263291
Geração 6: melhor y = 0.7500025195263291
Geração 7: melhor y = 0.7500025195263291
Geração 8: melhor y = 0.750002519526329